In [1]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout

def build_cnn_model(n_steps, n_features):
    model = Sequential([
        # 1. Feature Extraction: kernel_size=3 looks at 3-day windows
        Conv1D(filters=64, kernel_size=3, activation='relu', 
               padding='causal', input_shape=(n_steps, n_features)),
        MaxPooling1D(pool_size=2),
        
        # 2. Pattern Recognition: kernel_size=2 looks at higher-level patterns
        Conv1D(filters=32, kernel_size=2, activation='relu', padding='causal'),
        MaxPooling1D(pool_size=2),
        
        Flatten(),
        
        # 3. Decision Making
        Dense(50, activation='relu'),
        Dropout(0.2),
        Dense(1) # Predicts the next-day return or price
    ])
    
    model.compile(optimizer='adam', loss='huber')
    return model

In [2]:
from tensorflow.keras.layers import LSTM, TimeDistributed

def build_hybrid_model(n_steps, n_features):
    model = Sequential([
        # CNN layers to find local patterns in indicators (RSI, MACD, etc.)
        Conv1D(filters=64, kernel_size=3, activation='relu', 
               padding='causal', input_shape=(n_steps, n_features)),
        Conv1D(filters=64, kernel_size=3, activation='relu', padding='causal'),
        MaxPooling1D(pool_size=2),
        
        # LSTM layer to process the sequences extracted by the CNN
        LSTM(100, return_sequences=False),
        Dropout(0.3),
        
        Dense(25, activation='relu'),
        Dense(1)
    ])
    
    model.compile(optimizer='adam', loss='huber')
    return model

In [5]:
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout, Flatten

# 1. Load Data
ticker = "AAPL"
data = yf.download(ticker, start="2020-01-01", end="2024-01-01")

# 2. Feature Engineering (Keep your existing indicator logic here)
def add_indicators(df):
    df['Returns'] = df['Close'].pct_change()
    df['Log_Returns'] = np.log(df['Close'] / df['Close'].shift(1))
    
    # RSI
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))
    
    # Moving Averages
    df['MA10'] = df['Close'].rolling(window=10).mean()
    df['MA50'] = df['Close'].rolling(window=50).mean()
    
    # Volatility
    df['Volatility'] = df['Log_Returns'].rolling(window=21).std()
    
    return df.dropna()

df = add_indicators(data)

# 3. Preprocessing
# We predict 'Log_Returns' to avoid the price-lag trap
# 3. Preprocessing
features = ['Log_Returns', 'RSI', 'MA10', 'MA50', 'Volatility', 'Volume']
target = 'Log_Returns'

scaler = StandardScaler()
scaled_data = scaler.fit_transform(df[features])

# window_size defines how many days the model "looks back"
window_size = 30 

# Pass the FULL target values array so indexing remains aligned
X, y = create_sequences(scaled_data, df[target].values, window_size)

# Split (80/20)
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f"X_train shape: {X_train.shape}") # Should be (N, 30, 6)
print(f"y_train shape: {y_train.shape}") # Should be (N,)
# Split (80/20)
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# 4. CNN-LSTM Hybrid Model
model = Sequential([
    # CNN layer to extract "shapes" from indicators
    Conv1D(filters=64, kernel_size=3, activation='relu', 
           padding='causal', input_shape=(X_train.shape[1], X_train.shape[2])),
    MaxPooling1D(pool_size=2),
    
    # LSTM to handle the temporal sequence of the shapes found by CNN
    LSTM(50, return_sequences=False),
    Dropout(0.2),
    
    Dense(25, activation='relu'),
    Dense(1) # Predicts next-day Log Return
])

model.compile(optimizer='adam', loss='huber')

# 5. Training
history = model.fit(X_train, y_train, epochs=50, batch_size=32, 
                    validation_data=(X_test, y_test), verbose=1)

# 6. Evaluation
# .flatten() ensures the model output is a simple list of numbers (N,)
predictions_log_returns = model.predict(X_test).flatten()

# 1. Get the "Price of the Previous Day" for each test point
# Use .values.flatten() to ensure this is a 1D array
last_prices = df['Close'].values[split + window_size - 1 : -1].flatten()

# 2. Get the "Actual Price of the Target Day"
actual_prices = df['Close'].values[split + window_size:].flatten()

# 3. Calculate Predicted Price: P_t+1 = P_t * exp(return)
# Now that both are 1D, this will result in a 1D array of 186 values
predicted_prices = last_prices * np.exp(predictions_log_returns)

# 4. Metrics - Now the shapes will match perfectly (186, ) vs (186, )
mae = mean_absolute_error(actual_prices, predicted_prices)

# Directional Accuracy (comparing signs of returns)
# Ensure y_test is also flat
y_test_flat = y_test.flatten()
directional_acc = np.mean(np.sign(predictions_log_returns) == np.sign(y_test_flat))

print(f"\n--- {ticker} Evaluation ---")
print(f"MAE: ${mae:.2f}")
print(f"Directional Accuracy: {directional_acc:.2%}")

[*********************100%***********************]  1 of 1 completed

X_train shape: (741, 30, 6)
y_train shape: (741,)
Epoch 1/50



/Users/ttt/Downloads/DL4AI-240162-project/.venv/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0041 - val_loss: 2.2806e-04
Epoch 2/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0011 - val_loss: 7.1795e-04
Epoch 3/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 7.6070e-04 - val_loss: 5.0013e-04
Epoch 4/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 6.1121e-04 - val_loss: 2.3309e-04
Epoch 5/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 4.9447e-04 - val_loss: 1.3202e-04
Epoch 6/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 4.2316e-04 - val_loss: 1.7939e-04
Epoch 7/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3.9400e-04 - val_loss: 1.8845e-04
Epoch 8/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 3.5349e-04 - val_loss: 2.3557e-04
Epoch 9/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 3.4752e-04 - val_loss: 1.8196e-04
Epoch 10/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.7921e-04 - val_loss: 2.2684e-04
Epoch 11/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.9699e-04 - val_loss: 2.8584e-04
Epoch